## LLM Labeling. --By Melody 11-23version

Basic Logic:
1. use Gemini API key to first label the random 50 reviews from the whole inapproproate data set and make some polishment for the labels' instruction/rule/definition 

2. and then apply the latest version of the labels' instruction/rule/definition to the whole inappropriate data set.

Same workflow as HW5, with one difference: **a review can have several labels**.

| Step | What happens |
|---|---|
| 0 | Settings, Gemini connection, labels and definitions |
| 1 | Sample 50 reviews from the team's latest CSV and export them for hand labeling |
| 2 | Gemini labels the same 50 with the v1 instruction |
| 3 | Accuracy, per-label precision/recall, and the reviews where Gemini and the hand labels differ |
| 4 | The group polishes the definitions (v2) |
| 5 | Label **all** reviews with v2 and save the final CSV with the LLM labels |
| 6 | Use the v2 for in depth data visualization for the next step |


This version is written for the team's latest file, `inappropriate_reviews_llm_labels.csv`:

| Column | What it is | Sent to Gemini? |
|---|---|---|
| `review_id` | unique id | no (only used to match answers) |
| `book_id` | book id | no |
| `review_text` | the review | **yes, this is the only thing Gemini sees** |
| `labels` | the team's hand labels | **no**, used only to score Gemini afterwards |
| `notes` | the team's notes | **no** |
| `llm_labels` | **filled in by this notebook** | – |
| `llm_reasoning` | **filled in by this notebook** | – |


### 0-1 Settings and connection

In [1]:
import ast, json, os, time
from pathlib import Path

import numpy as np
import pandas as pd

# ====== CHANGE THESE ======
DATA_PATH = Path("data/inappropriate_reviews_llm_labels.csv")   # the team's latest CSV
ID_COL = "review_id"
TEXT_COL = "review_text"      # the ONLY column sent to Gemini
HAND_COL = "labels"           # hand labels: used for scoring only, never sent to Gemini
OUT_DIR = Path("llm_output")  # everything this notebook creates goes here
SAMPLE_SIZE = 50
SEED = 211
SLEEP_SECONDS = 6             # wait between calls to stay under the free tier's limit per minute
# ==========================

OUT_DIR.mkdir(exist_ok=True)

In [4]:
# Connect to Gemini (same as HW5). The key is read from .env and never printed.
import logging

from dotenv import load_dotenv, find_dotenv
from google import genai
from google.genai import errors, types

load_dotenv(find_dotenv(usecwd=True))
key = os.environ.get("GEMINI_API_KEY", "")
if not key or key.startswith("paste-"):
    raise ValueError("Add GEMINI_API_KEY to your .env file, then run this cell again.")

client = genai.Client(api_key=key)
GEMINI_MODEL = "gemini-3.5-flash-lite"  # use the replacement model if bCourses announces one

logging.getLogger("google_genai.types").setLevel(logging.ERROR)
print("Connected to Gemini.")

Connected to Gemini.


In [5]:
import sys
print(sys.executable)

/Users/ashley/compss-211a/.venv/bin/python


### 0-2 Labels and definitions

The rules come from the team's Goodreads Project Outline. **All definitions live in `LABEL_RULES` and `EXTRA_RULES`**, and the instruction text is built from them

***a problem:The hand labels in the CSV also contain `preachy` and `sexuality`, which are not in the outline's list of 16 labels, so Gemini does not know them.


In [7]:
LABEL_RULES = {
    "age-appropriateness": "The review says the book is not appropriate for children. Do NOT use it when the reviewer only says they themselves are too old for the book.",
    "religion": "Religion or spirituality is part of the complaint (use only when it is part of the complaint).",
    "racism": "The complaint is about race, racism (including so-called reverse racism), racist or ethnic stereotyping, or xenophobia.",
    "body-shaming": "The complaint is about body-shaming, body image, or fatphobia.",
    "bad-influence": "The book has a bad influence on kids, or teaches kids wrong or immoral behaviors and lessons.",
    "scary/dark": "The book is too scary, dark, disturbing, creepy, or described with similar words.",
    "illustrations": "The book's illustrations are part of the complaint (use only when illustrations are part of the complaint).",
    "abuse": "The complaint is about the book's portrayal of physical, psychological, or sexual abuse (the review explicitly mentions abuse).",
    "ableism": "The complaint is about the treatment of disabled people, including physical disabilities and developmental disabilities such as autism.",
    "non-factual": "The book's content gives incorrect information.",
    "sexism": "The complaint is about sexism, misogyny, patriarchy, or a negative or stereotypical portrayal of women and girls.",
    "classism": "The complaint is about classism, wealth superiority, or elitism.",
    "violence": "The complaint is about violence or any violent or harmful scenario.",
    "witchcraft": "Witchcraft is part of the complaint (use only when witchcraft is part of the complaint).",
    "crude-language": "The complaint is about crude, suggestive, sexual, or vulgar language or illustrations.",
    "drugs/alcohol": "The complaint is about the portrayal of drugs and/or alcohol in the book.",
    "preachy": "The complaint mentions the book being preachy.",
    "sexuality": "The complaint mentions sexuality, sexual orientation, or related words such as queer, gay, lesbian, homophobic, and homosexual.",
}
LABELS = list(LABEL_RULES)

EXTRA_RULES = [
    "Label by the topic of the complaint. Give EVERY label whose rule is met. Give none if no rule is met.",
    "Some complaints are inappropriate regardless of age (for example racist content). Do not add age-appropriateness for those unless the reviewer also says the book is not appropriate for children.",
    "bad-influence and drugs/alcohol are always about a young audience: whenever you give either one, also give age-appropriateness.",
    "Base the labels only on what the reviewer actually says. Do not infer a complaint that is not in the text.",
]


def build_system(rules, extra):
    lines = [
        "You are labeling one-star Goodreads reviews of children's books that were flagged as possibly complaining about inappropriateness.",
        "Decide which of the labels below apply to the review. A review can have several labels, one label, or none.",
        "",
        "Labels:",
    ]
    lines += [f"- {name}: {rule}" for name, rule in rules.items()]
    lines += ["", "Rules:"] + [f"- {r}" for r in extra]
    lines += ["", "Return llm_labels (a list of label names copied exactly as written above; an empty list if none apply) and llm_reasoning (one short sentence)."]
    return "\n".join(lines)


SYSTEM_V1 = build_system(LABEL_RULES, EXTRA_RULES)


def build_schema(labels):
    """The reply format Gemini must follow."""
    return {
        "type": "object",
        "properties": {
            "llm_labels": {"type": "array", "items": {"type": "string", "enum": list(labels)}},
            "llm_reasoning": {"type": "string"},
        },
        "required": ["llm_labels", "llm_reasoning"],
    }


SCHEMA = build_schema(LABELS)   # if you add labels later, rebuild this from the new LABELS
print(SYSTEM_V1)

You are labeling one-star Goodreads reviews of children's books that were flagged as possibly complaining about inappropriateness.
Decide which of the labels below apply to the review. A review can have several labels, one label, or none.

Labels:
- age-appropriateness: The review says the book is not appropriate for children. Do NOT use it when the reviewer only says they themselves are too old for the book.
- religion: Religion or spirituality is part of the complaint (use only when it is part of the complaint).
- racism: The complaint is about race, racism (including so-called reverse racism), racist or ethnic stereotyping, or xenophobia.
- body-shaming: The complaint is about body-shaming, body image, or fatphobia.
- bad-influence: The book has a bad influence on kids, or teaches kids wrong or immoral behaviors and lessons.
- scary/dark: The book is too scary, dark, disturbing, creepy, or described with similar words.
- illustrations: The book's illustrations are part of the compla

#### 1. Load the table and sample 50 reviews

In [8]:
# keep_default_na=False keeps empty cells as "" instead of NaN, so the table is written back exactly as it was read.
DATA_PATH = Path("inappropriate_reviews_llm_labels.csv")

data = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)
ORIGINAL_COLUMNS = list(data.columns)

for col in (ID_COL, TEXT_COL, HAND_COL, "llm_labels", "llm_reasoning"):
    assert col in data.columns, f"Column {col!r} is missing. The columns are: {ORIGINAL_COLUMNS}"
assert data[ID_COL].is_unique, "review_id is not unique"
assert (data[TEXT_COL].str.strip() != "").all(), "Some reviews have no text"
print(data.shape, ORIGINAL_COLUMNS)

# What Gemini is allowed to see: ONLY the id and the review text. labels and notes are not in this table at all.
to_label = data[[ID_COL, TEXT_COL]].rename(columns={ID_COL: "id", TEXT_COL: "text"})

sample = to_label.sample(n=SAMPLE_SIZE, random_state=SEED)
print(len(to_label), "reviews in total;", len(sample), "in the sample")

(163, 7) ['review_id', 'book_id', 'review_text', 'labels', 'notes', 'llm_labels', 'llm_reasoning']
163 reviews in total; 50 in the sample


In [9]:
# The hand labels ("['age-appropriateness', 'racism']" -> a set of label names). Used ONLY to score Gemini.
def parse_hand(cell):
    try:
        items = ast.literal_eval(cell)
        assert isinstance(items, (list, tuple))
    except (ValueError, SyntaxError, AssertionError):
        items = str(cell).replace("[", "").replace("]", "").replace("'", "").split(",")
    return {str(x).strip().lower() for x in items if str(x).strip()}


hand_sets = {rid: parse_hand(cell) for rid, cell in zip(data[ID_COL], data[HAND_COL])}

# Hand labels that Gemini has no definition for (it can never predict them):
extra_hand_labels = sorted({lab for s in hand_sets.values() for lab in s} - set(LABELS))
print("Hand labels not in LABELS:", extra_hand_labels or "none")

Hand labels not in LABELS: none


#### 2. Gemini labels the 50 with the v1 instruction

In [10]:
# Supplied (same as HW5, changed to many labels per review).
# Errors worth trying again, because they go away on their own: 429 = too many calls in one minute, 503 = server busy.
RETRY_CODES = {429, 503}


def ask_gemini(prompt, system, schema, max_tries=4):
    """Send one prompt with this system instruction and schema. Return the labels and the reasoning. Try again after a temporary error."""
    wait = 10  # seconds to wait before the first retry
    for attempt in range(1, max_tries + 1):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system,
                    response_mime_type="application/json",
                    response_schema=schema,
                ),
            )
            time.sleep(SLEEP_SECONDS)
            return json.loads(response.text)
        except errors.APIError as error:
            if error.code not in RETRY_CODES or attempt == max_tries:
                raise  # stop and show the error
            print(f"Error {error.code}. Waiting {wait} seconds, then trying again.")
            time.sleep(wait)
            wait = wait * 2  # wait longer each time: 10, 20, 40 seconds


def label_texts(texts, system, schema, path):
    """Ask Gemini about every review in texts (columns id and text only). Save each answer to path right away.
    Reviews that are already in the file are skipped, so you can run it again after an interruption."""
    done = set(pd.read_csv(path, dtype={"id": str})["id"]) if path.exists() else set()
    for row in texts.itertuples():
        if row.id in done:
            continue
        reply = ask_gemini(f"Review: {row.text}", system, schema)
        labels = list(dict.fromkeys(lab for lab in reply.get("llm_labels", []) if lab in schema_labels(schema)))  # valid names, no repeats
        out = pd.DataFrame([{
            "id": row.id,
            "llm_labels": str(labels),                      # same format as the hand labels: "['a', 'b']", or "[]"
            "llm_reasoning": reply.get("llm_reasoning", ""),
            "model": GEMINI_MODEL,
        }])
        out.to_csv(path, mode="a", header=not path.exists(), index=False)
        done.add(row.id)
        print(len(done), "of", len(texts), "done")
    results = pd.read_csv(path, dtype=str, keep_default_na=False)
    return results[results["id"].isin(texts["id"])]


def schema_labels(schema):
    return schema["properties"]["llm_labels"]["items"]["enum"]

In [17]:
# Label the 50 with the v1 instruction (50 calls, a few minutes). If it stops, run this cell again: it skips the reviews already done.
v1_results = label_texts(sample, SYSTEM_V1, SCHEMA, OUT_DIR / "llm_v1_sample50_raw.csv")
len(v1_results)   # Check: 50

50

In [18]:
def fill_columns(table, results, only_ids=None):
    """Return the team's table (same columns, same order) with llm_labels and llm_reasoning filled from results."""
    out = table[ORIGINAL_COLUMNS].copy()
    out["llm_labels"] = out[ID_COL].map(dict(zip(results["id"], results["llm_labels"]))).fillna("")
    out["llm_reasoning"] = out[ID_COL].map(dict(zip(results["id"], results["llm_reasoning"]))).fillna("")
    if only_ids is not None:
        out = out[out[ID_COL].isin(only_ids)]
    return out


v1_filled = fill_columns(data, v1_results, only_ids=sample["id"])
v1_filled.to_csv(OUT_DIR / "inappropriate_reviews_llm_v1_sample50.csv", index=False)
print("Saved the 50 sampled reviews with llm_labels and llm_reasoning filled in:", v1_filled.shape)
v1_filled[[ID_COL, HAND_COL, "llm_labels", "llm_reasoning"]].head()

Saved the 50 sampled reviews with llm_labels and llm_reasoning filled in: (50, 7)


,review_id,labels,llm_labels,llm_reasoning
1,acf5e394790f363eec00bb12df9d806a,"['bad-influence', 'age-appropriateness']","['age-appropriateness', 'bad-influence', 'abuse']",The reviewer complains that the book depicts i...
7,a131a93f8b74e838733a9e87de53b76a,['age-appropriateness'],"['age-appropriateness', 'crude-language']",The reviewer complains that the game spin the ...
8,44636b23c32494e0d823b90d9764b200,"['scary/dark', 'age-appropriateness', 'illustr...","['illustrations', 'scary/dark']",The reviewer complains about the dark illustra...
10,614dbe3c62c83d77269b470598eb23b6,"['racism', 'age-appropriateness']","['racism', 'age-appropriateness']",The reviewer explicitly complains about racist...
31,72e743a54a154ad6b297c03646c7c4f4,"['bad-influence', 'age-appropriateness', 'crud...","['age-appropriateness', 'crude-language', 'bad...",The reviewer complains about inappropriate lan...


#### 3. Accuracy and disagreements

With several labels per review there are a few ways to measure agreement, so we report them all:

- **Exact-match accuracy:** the share of reviews where Gemini's set of labels is *exactly* the hand-labeled set. This is the strictest number.
- **Per-label precision and recall:** for each label, `precision` = of the reviews Gemini gave this label, how many did the hand labels also have; `recall` = of the reviews with this hand label, how many did Gemini find.
  A *false positive* is a label Gemini added that the hand labels do not have; a *false negative* is a hand label that Gemini missed.
- **Micro precision / recall / F1:** the same idea counted over all labels together.
- **Average overlap (Jaccard):** how much the two label sets overlap on average (1 = identical, 0 = nothing in common).

In [19]:
def to_set(cell):
    try:
        return {str(x) for x in ast.literal_eval(cell)} if str(cell).strip() else set()
    except (ValueError, SyntaxError):
        return set()


def evaluate(table, results):
    """Compare the hand labels in `table` with Gemini's labels in `results`. Returns one row per review, one row per label, and a summary."""
    t = table[[ID_COL, TEXT_COL, HAND_COL, "notes"]].merge(
        results[["id", "llm_labels", "llm_reasoning"]], left_on=ID_COL, right_on="id", how="inner")
    t["hand_set"] = [hand_sets[r] for r in t[ID_COL]]
    t["llm_set"] = t["llm_labels"].apply(to_set)
    t["exact_match"] = [h == g for h, g in zip(t["hand_set"], t["llm_set"])]
    t["missed_by_llm"] = [", ".join(sorted(h - g)) for h, g in zip(t["hand_set"], t["llm_set"])]     # false negatives
    t["extra_from_llm"] = [", ".join(sorted(g - h)) for h, g in zip(t["hand_set"], t["llm_set"])]    # false positives
    t["jaccard"] = [1.0 if not (h | g) else len(h & g) / len(h | g) for h, g in zip(t["hand_set"], t["llm_set"])]

    rows = []
    for lab in LABELS + extra_hand_labels:      # labels Gemini cannot know are included, so their misses are visible
        h = np.array([lab in s for s in t["hand_set"]])
        g = np.array([lab in s for s in t["llm_set"]])
        tp, fp, fn = int((h & g).sum()), int((~h & g).sum()), int((h & ~g).sum())
        rows.append({
            "label": lab, "hand_count": int(h.sum()), "llm_count": int(g.sum()),
            "true_pos": tp, "false_pos": fp, "false_neg": fn,
            "precision": tp / (tp + fp) if tp + fp else np.nan,
            "recall": tp / (tp + fn) if tp + fn else np.nan,
        })
    per_label = pd.DataFrame(rows).set_index("label")

    tp, fp, fn = per_label["true_pos"].sum(), per_label["false_pos"].sum(), per_label["false_neg"].sum()
    p = tp / (tp + fp) if tp + fp else np.nan
    r = tp / (tp + fn) if tp + fn else np.nan
    summary = {
        "n_reviews": len(t),
        "exact_match_accuracy": t["exact_match"].mean(),
        "average_overlap_jaccard": t["jaccard"].mean(),
        "micro_precision": p,
        "micro_recall": r,
        "micro_f1": 2 * p * r / (p + r) if p + r else np.nan,
    }
    return t, per_label, summary

In [20]:
v1_eval, v1_per_label, v1_summary = evaluate(data, v1_results)

print(pd.Series(v1_summary).round(3).to_string())
v1_per_label.round(2)

n_reviews                  50.000
exact_match_accuracy        0.500
average_overlap_jaccard     0.785
micro_precision             0.795
micro_recall                0.907
micro_f1                    0.847


,hand_count,llm_count,true_pos,false_pos,false_neg,precision,recall
label,,,,,,,
age-appropriateness,41,40,38,2,3,0.95,0.93
religion,1,1,1,0,0,1.00,1.00
racism,7,8,7,1,0,0.88,1.00
body-shaming,1,1,1,0,0,1.00,1.00
bad-influence,11,13,10,3,1,0.77,0.91
scary/dark,16,19,16,3,0,0.84,1.00
illustrations,5,11,4,7,1,0.36,0.80
abuse,1,2,1,1,0,0.50,1.00
ableism,1,1,1,0,0,1.00,1.00


In [21]:
# The reviews where Gemini's labels are NOT exactly the hand labels. Read these as a group.
pd.set_option("display.max_colwidth", None)

v1_disagreements = v1_eval[~v1_eval["exact_match"]][
    [ID_COL, TEXT_COL, HAND_COL, "llm_labels", "missed_by_llm", "extra_from_llm", "llm_reasoning", "notes"]
]
v1_disagreements.to_csv(OUT_DIR / "v1_disagreements.csv", index=False)   # share this file with the group
print(len(v1_disagreements), "of", len(v1_eval), "reviews differ")
v1_disagreements.head(5)

25 of 50 reviews differ


,review_id,review_text,labels,llm_labels,missed_by_llm,extra_from_llm,llm_reasoning,notes
0,acf5e394790f363eec00bb12df9d806a,"While my kindergartner enjoyed the mystery of this book, and would probably be interested in more stories, it was a real snoozer for mom, and parts of it made me uncomfortable. Nonetheless, it remains a highly recommended work for the age group, and was so successful the author released an entire series that I shall try to avoid in future purchases. I suppose not all books for children can be as wonderful as ""The Little House on the Prairie"" series, which I would rather read. \n Is the world so saccharine? Four children loose their parents and start off on their own, yet we see little to no mourning. They escape from a couple who has given them lodging for a night with intention to split them up and put the older kids to work for them, yet the plot continues without any real sense of danger past or future. Perfect shelter appears, with water and a nearby dump that accommodates all their needs -- though they do have chipped cups, poor kids (hear the sarcasm this book brings out in me?). A good paying job with a trustworthy employer is found the first time out, and any person who arouses a hint of discomfort turns out to be a friend. Of course, a wealthy relative eager to meet their every desire is found at the end in order to ensure their unshakable happiness. \n If running away were so easy, I fear most children would do so under far less provocation than the Box Car Children! \n I also found that times have changed. I was uncomfortable by references to strangers asking the children to sit on their lap, touching them and cuddling them. Clearly the tone of the book is wholesome these were to be signs of affection and reassurance, but the book as a whole struck me as weird Twilight Zone material. \n On the positive side, the children are loyal to each other and care for each other. The older children willingly share a greater part of the burden in caring for the younger. They take initiative to set up and improve their home. They all work hard. And there is nothing to be afraid of. \n Perhaps your child is a fragile soul who needs to be protected for a while longer from the real world. Or, you are a parent who doesn't want your child reading inappropriate material. Maybe this would be a good early reader for children to read themselves. In all of these cases, this might be a good book for you. \n But beware the thinking that this book is anywhere near real life, which might lead your child to trust strangers and assume all their needs will be met serendipitously.","['bad-influence', 'age-appropriateness']","['age-appropriateness', 'bad-influence', 'abuse']",,abuse,"The reviewer complains that the book depicts inappropriate stranger interactions resembling abuse, gives kids a bad influence by making running away seem easy, and mentions the book might not be appropriate for some children.",
1,a131a93f8b74e838733a9e87de53b76a,"The story is weak, even for a children's book, but that doesn't bother me. What does is that Marc Brown somehow thinks spin the bottle is an appropriate present for a child. Completely inappropriate for the target age.",['age-appropriateness'],"['age-appropriateness', 'crude-language']",,crude-language,The reviewer complains that the game spin the bottle is inappropriate for the target age group of children.,
2,44636b23c32494e0d823b90d9764b200,"The illustrations are very dark and the storyline really doesn't play well with preschool audiences so to be honest, I really don't see this book being very useful for storytelling. The theme of the book focuses on a rabbit and squirrel that have lived side by side for years yet never talked to each other. When they start to find their gardens being destroyed, they automatically start blaming the other. Without even trying to find out the facts, they start warring. Even when they discover the real reason the gardens are dissapearing, they're h

**For each disagreement ask:** is Gemini wrong, are the hand labels wrong, or is the definition unclear?
Look first at labels with many false positives or false negatives in the table above, and at labels whose definitions overlap
(for example `scary/dark` and `violence`, or `bad-influence` and `age-appropriateness`).
The `notes` column is shown here only to help the group read the cases; Gemini never sees it.



#### 4-1. The group polishes the definitions (v2)

Edit the definitions below after the group meeting. 

In [14]:
# V2 definitions, copied from the V2 column of the team's Goodreads Project Outline (updated 11/24).
LABEL_RULES_V2 = {
    "age-appropriateness": "Flag when the review talks about the book not being appropriate for children, young audiences, or the targeted demographic. Do not flag reviews that mention a reviewer being too old for a book, as that does not indicate inappropriateness in the context of our research question.",
    "religion": "Flag when the review mentions religion or spirituality as part of the complaint.",
    "racism": "Flag review when the complaint is about race, racism, racist/ethnic stereotyping, and xenophobia.",
    "body-shaming": "Flag review when the complaint is about body-shaming, body-image, and fatphobia.",
    "bad-influence": "Flag review if it talks about the book having a bad influence on kids/youth or teaching kids/youth wrong or immoral behaviors and lessons.",
    "scary/dark": "Flag review when the complaint is about a book being too scary, dark, disturbing, creepy, or other similar words.",
    "illustrations": "Flag review when a book's illustrations or pictures are mentioned to be inappropriate, offensive, or not age-appropriate.",
    "abuse": "Flag review when the complaint is about a book's portrayal of physical, psychological, and/or sexual abuse.",
    "ableism": "Flag review when the complaint is about treatment of disabled people including physical disabilities and developmental disabilities like autism.",
    "non-factual": "Flag review when it mentions a book's content explicitly giving incorrect information.",
    "sexism": "Flag review when the complaint is about sexism, misogyny, patriarchy, or negative/stereotypical portrayal of women/girls.",
    "classism": "Flag review if the complaint is about classism, wealth superiority, and elitism.",
    "violence": "Flag review if the complaint is about violence or any violent/harmful scenario.",
    "witchcraft": "Flag review when witchcraft is part of the complaint.",
    "crude-language": "Flag review when the complaint is about the book having crude, suggestive, sexual, or vulgar writing/illustrations.",
    "drugs/alcohol": "Flag review when the complaint is about portrayal of drugs, alcohol, drug addicts, drug addiction, and alcoholics in the book.",
    "preachy": "Flag review when the complaint mentions the book being preachy.",
    "sexuality": "Flag review when the complaint mentions sexuality, sexual orientation, or other related words such as queer, gay, lesbian, homophobic, and homosexual.",
}
assert list(LABEL_RULES_V2) == LABELS, "V2 labels must match the V1 labels"
EXTRA_RULES_V2 = list(EXTRA_RULES)

# Examples of further edits (delete the # to use):
# LABEL_RULES_V2["scary/dark"] = "The book is too scary, dark, disturbing or creepy, but NOT when the complaint is only about violent events (use violence for those)."
# EXTRA_RULES_V2.append("If a review fits both scary/dark and violence, give both.")

LABELS_V2 = list(LABEL_RULES_V2)
SYSTEM_V2 = build_system(LABEL_RULES_V2, EXTRA_RULES_V2)
SCHEMA_V2 = build_schema(LABELS_V2)
print("v2 instruction differs from v1:", SYSTEM_V2 != SYSTEM_V1, "| labels:", len(LABELS_V2))

v2 instruction differs from v1: True | labels: 18


In [22]:
# Optional: score v2 on the same 50. This number is too optimistic, because the definitions were polished
# after reading Gemini's mistakes on these same 50 reviews. Use it only to check that nothing got worse.
v2_results = label_texts(sample, SYSTEM_V2, SCHEMA_V2, OUT_DIR / "llm_v2_sample50_raw.csv")
v2_eval, v2_per_label, v2_summary = evaluate(data, v2_results)

pd.DataFrame({"v1": v1_summary, "v2 (optimistic)": v2_summary}).round(3)

,v1,v2 (optimistic)
n_reviews,50.000,50.000
exact_match_accuracy,0.500,0.620
average_overlap_jaccard,0.785,0.838
micro_precision,0.795,0.846
micro_recall,0.907,0.925
micro_f1,0.847,0.884


#### 4-2 Explaination: Evaluation metrics

We compare Gemini's labels with the team's hand labels for the same reviews.
Because a review can have several labels, each review is compared as a set of labels.

- **Exact-match accuracy**: the share of reviews for which Gemini's set of labels is identical to the hand-labeled set.
  A review counts as correct only if Gemini gave every hand label and no extra label. This is the strictest metric.
- **Average Jaccard overlap**: for each review, the number of labels both sets share divided by the number of labels in either set,
  averaged over all reviews. It ranges from 0 (no overlap) to 1 (identical) and gives partial credit when Gemini is close but not exact.
- **True positive (TP), false positive (FP), false negative (FN)**: counted over every review-label pair.
  A TP is a label that Gemini and the hand labels both assigned. An FP is a label Gemini assigned that the hand labels did not.
  An FN is a hand label that Gemini did not assign.
- **Micro precision** = TP / (TP + FP). Of all labels Gemini assigned, the share that matches a hand label.
  Low precision means Gemini adds labels that humans did not.
- **Micro recall** = TP / (TP + FN). Of all hand labels, the share that Gemini also assigned.
  Low recall means Gemini misses labels that humans gave.
- **Micro F1** = 2 × precision × recall / (precision + recall). The harmonic mean of precision and recall,
  which is high only when both are high. "Micro" means that counts are pooled across all labels before the ratios are computed,
  so frequent labels such as age-appropriateness weigh more.
- **Per-label precision and recall**: the same formulas computed for one label at a time,
  used to find which label definitions need to be clarified.

#### 4-3 Finding Summary

Gemini recovers most hand-labeled topics (micro recall 0.93) and over-labels less after the definitions were refined (micro precision 0.80 → 0.85). These scores are optimistic because the definitions were tuned on the same sample, so we report the held-out result as the main estimate.

In [ ]:
# Fill llm_labels and llm_reasoning into the team's table. Same columns, same order, nothing else changed.
final = fill_columns(data, all_results)
assert (final["llm_reasoning"] != "").all(), "Some reviews are not labeled yet: run the previous cell again."
assert list(final.columns) == ORIGINAL_COLUMNS
assert final[[ID_COL, "book_id", TEXT_COL, HAND_COL, "notes"]].equals(data[[ID_COL, "book_id", TEXT_COL, HAND_COL, "notes"]])

FINAL_CSV = OUT_DIR / "inappropriate_reviews_llm_labels_filled.csv"
final.to_csv(FINAL_CSV, index=False)
print("Saved", FINAL_CSV, final.shape)
final[[ID_COL, HAND_COL, "llm_labels", "llm_reasoning"]].head()

#### 5. Label all reviews with the final instruction

This sends **every** review to Gemini, one call each, about 7 seconds per review plus any retries. Check the count first.
Each answer is saved right away, so if it stops (for example after repeated 503 errors) just run the cell again.

In [23]:
FINAL_SYSTEM, FINAL_SCHEMA = SYSTEM_V2, SCHEMA_V2     # the polished definitions
print(len(to_label), "reviews will be labeled, about", round(len(to_label) * (SLEEP_SECONDS + 1) / 60), "minutes")

163 reviews will be labeled, about 19 minutes


In [24]:
all_results = label_texts(to_label, FINAL_SYSTEM, FINAL_SCHEMA, OUT_DIR / "llm_all_v2_raw.csv")
len(all_results)   # Check: the same as the number of reviews above

1 of 163 done
2 of 163 done
3 of 163 done
4 of 163 done
5 of 163 done
6 of 163 done
7 of 163 done
8 of 163 done
9 of 163 done
10 of 163 done
Error 503. Waiting 10 seconds, then trying again.
11 of 163 done
12 of 163 done
13 of 163 done
14 of 163 done
15 of 163 done
16 of 163 done
17 of 163 done
18 of 163 done
19 of 163 done
20 of 163 done
21 of 163 done
22 of 163 done
23 of 163 done
24 of 163 done
25 of 163 done
26 of 163 done
27 of 163 done
28 of 163 done
29 of 163 done
30 of 163 done
31 of 163 done
32 of 163 done
33 of 163 done
34 of 163 done
35 of 163 done
36 of 163 done
37 of 163 done
38 of 163 done
39 of 163 done
40 of 163 done
41 of 163 done
42 of 163 done
43 of 163 done
44 of 163 done
45 of 163 done
Error 503. Waiting 10 seconds, then trying again.
46 of 163 done
Error 503. Waiting 10 seconds, then trying again.
47 of 163 done
48 of 163 done
49 of 163 done
50 of 163 done
51 of 163 done
52 of 163 done
53 of 163 done
54 of 163 done
55 of 163 done
56 of 163 done
57 of 163 done
58 o

163

In [25]:
# Fill llm_labels and llm_reasoning into the team's table. Same columns, same order, nothing else changed.
final = fill_columns(data, all_results)
assert (final["llm_reasoning"] != "").all(), "Some reviews are not labeled yet: run the previous cell again."
assert list(final.columns) == ORIGINAL_COLUMNS
assert final[[ID_COL, "book_id", TEXT_COL, HAND_COL, "notes"]].equals(data[[ID_COL, "book_id", TEXT_COL, HAND_COL, "notes"]])

FINAL_CSV = OUT_DIR / "inappropriate_reviews_llm_labels_filled.csv"
final.to_csv(FINAL_CSV, index=False)
print("Saved", FINAL_CSV, final.shape)
final[[ID_COL, HAND_COL, "llm_labels", "llm_reasoning"]].head()

Saved llm_output/inappropriate_reviews_llm_labels_filled.csv (163, 7)


,review_id,labels,llm_labels,llm_reasoning
0,8271a9e037ebe67dab6376fe498dbcf9,"['religion', 'age-appropriateness']","['age-appropriateness', 'religion']",The reviewer complains about theological errors concerning the Christian faith and states the age designation for children is inappropriate.
1,acf5e394790f363eec00bb12df9d806a,"['bad-influence', 'age-appropriateness']","['age-appropriateness', 'bad-influence', 'abuse']","The reviewer expresses concern that the book might lead children to inappropriately trust strangers and run away, mentioning uncomfortable physical interactions with strangers."
2,dafeeb582114e849cb971e71f97f660a,"['religion', 'age-appropriateness']","['religion', 'age-appropriateness']",The review discusses a children's Bible and complains about certain religious concepts not being age-appropriate for preschoolers.
3,206b8919325f9737f91286d6c9e3e0a9,"['age-appropriateness', 'illustrations']","['age-appropriateness', 'illustrations', 'crude-language']",The reviewer complains that the humor and illustrations of undergarments are too adult and inappropriate for children.
4,e0337f5711a585a85a238d53f122ed69,['racism'],"['racism', 'bad-influence', 'age-appropriateness']",The reviewer complains about racism in the book and expresses concern that the content will negatively influence their young child.


In [26]:
# A fair check of v2: the reviews that were NOT in the 50 used to polish the definitions.
held_out = data[~data[ID_COL].isin(sample["id"])]
ho_eval, ho_per_label, ho_summary = evaluate(held_out, all_results)

print(pd.Series(ho_summary).round(3).to_string())
ho_per_label.round(2)

n_reviews                  113.000
exact_match_accuracy         0.496
average_overlap_jaccard      0.749
micro_precision              0.749
micro_recall                 0.915
micro_f1                     0.824


,hand_count,llm_count,true_pos,false_pos,false_neg,precision,recall
label,,,,,,,
age-appropriateness,73,88,71,17,2,0.81,0.97
religion,9,9,8,1,1,0.89,0.89
racism,18,19,18,1,0,0.95,1.00
body-shaming,3,3,3,0,0,1.00,1.00
bad-influence,20,34,17,17,3,0.50,0.85
scary/dark,23,30,21,9,2,0.70,0.91
illustrations,13,9,7,2,6,0.78,0.54
abuse,3,4,3,1,0,0.75,1.00
ableism,3,4,3,1,0,0.75,1.00


In [27]:
n = len(final)
llm_counts = pd.Series([lab for cell in final["llm_labels"] for lab in sorted(to_set(cell))]).value_counts()
hand_counts = pd.Series([lab for s in hand_sets.values() for lab in s]).value_counts()

topic_table = pd.DataFrame({"llm_reviews": llm_counts, "hand_reviews": hand_counts}).fillna(0).astype(int)
topic_table["llm_share"] = (topic_table["llm_reviews"] / n).round(3)
topic_table["hand_share"] = (topic_table["hand_reviews"] / n).round(3)
topic_table = topic_table.sort_values("llm_reviews", ascending=False)
topic_table.index.name = "label"
topic_table.to_csv(OUT_DIR / "topic_counts.csv")

print(n, "reviews;", sum(1 for c in final["llm_labels"] if not to_set(c)), "got no label from Gemini")
topic_table

163 reviews; 2 got no label from Gemini


,llm_reviews,hand_reviews,llm_share,hand_share
label,,,,
age-appropriateness,131,114,0.804,0.699
scary/dark,50,39,0.307,0.239
bad-influence,50,31,0.307,0.190
racism,27,25,0.166,0.153
crude-language,24,23,0.147,0.141
violence,23,15,0.141,0.092
illustrations,13,18,0.080,0.110
drugs/alcohol,12,10,0.074,0.061
religion,10,10,0.061,0.061
